# CHIIR 2027 proceedings runbook

Work down this notebook over the proceedings season, from acceptance to front matter. Each step says what to do at that point and holds the settings it needs. Start Jupyter from the repository root, so that the `proceedings` package can be imported.

> **Before you commit:** cell outputs contain author names and emails. Run `nbstripout --install` once in this repository (see the README), so outputs are removed automatically.

In [ ]:
YEAR = 2027
NAME = "CHIIR"

import logging
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

from proceedings import (CategoryRule, ConferenceConfig, PaperType, check_erights_csv,
                         check_xml_against_source, export_to_erights_csv, export_to_xml,
                         heading_previews, load_affiliation_mapping, load_easychair_data)

# Show only problems from the loader; this notebook prints the results itself
logging.basicConfig(level=logging.ERROR, format="%(message)s", stream=sys.stdout, force=True)

## 1. Get the data from EasyChair

Do this once decisions are final, and again each time you fix data in EasyChair. Either download works, and both give the same result:

| Option | Where in EasyChair | Save as |
|---|---|---|
| Excel workbook (needs a licence that allows it) | **Conference → Data download/Export → Excel** | `export.xlsx` |
| CSV pair | **Submissions → Download**: *All submissions (CSV)* and *Authors (CSV)* | `submission.csv`, `author.csv` |

Save the files **outside this repository**, because they contain personal data. To try the notebook without real data, run `python tests/fixtures/make_fixtures.py ~/CHIIR2027-demo` and point `DATA_DIR` at that folder.

In [ ]:
DATA_DIR = Path.home() / "CHIIR2027-proceedings"

EASYCHAIR_EXPORT = DATA_DIR / "export.xlsx"
# EASYCHAIR_EXPORT = (DATA_DIR / "submission.csv", DATA_DIR / "author.csv")

import proceedings
REPO_ROOT = Path(proceedings.__file__).resolve().parent.parent
assert REPO_ROOT not in [DATA_DIR.resolve(), *DATA_DIR.resolve().parents], "Keep EasyChair data outside the repository"

## 2. Decide what counts as accepted, and as which paper type

**Accepted decisions** must match EasyChair's decision labels exactly, ignoring case. Check the breakdown printed below: anything not listed is left out.

**Paper types** are the names ACM sees in eRights, in proceedings order. The prefix makes the event tracking number, e.g. `fp42` for submission 42. Confirm the type names with the ACM proceedings coordinator before uploading.

**Category rules** are checked from top to bottom and the first match wins. CHIIR's single submission track asks authors for a `(Category) …` form field, while DC, tutorials and workshops have their own tracks. If you add a track or form option in EasyChair, add a rule here. Any accepted paper that matches no rule is reported as an error in step 3.

In [ ]:
ACCEPTED_DECISIONS = ["accept", "probably accept"]

PAPER_TYPES = [
    PaperType(name="Full Paper",          prefix="fp", heading="Full Papers",         file_stem="Full-Papers"),
    PaperType(name="Short Paper",         prefix="sp", heading="Short Papers",        file_stem="Short-Papers"),
    PaperType(name="Demonstration Paper", prefix="dp", heading="Demonstrations",      file_stem="Demonstrations"),
    PaperType(name="Doctoral Consortium", prefix="dc", heading="Doctoral Consortium", file_stem="Doctoral-Consortium"),
    PaperType(name="Tutorial",            prefix="tp", heading="Tutorials",           file_stem="Tutorials"),
    PaperType(name="Workshop",            prefix="wk", heading="Workshops",           file_stem="Workshops"),
]

CATEGORY_RULES = [
    CategoryRule(field="form fields", contains=["(category) full paper", "(category) resource paper",
                                                "(category) perspective"],      paper_type="Full Paper"),
    CategoryRule(field="form fields", contains=["(category) demonstration"],   paper_type="Demonstration Paper"),
    CategoryRule(field="form fields", contains=["(category) short paper"],     paper_type="Short Paper"),
    CategoryRule(field="track name",  contains_all=["chiir", "dc"],            paper_type="Doctoral Consortium"),
    CategoryRule(field="track name",  contains_all=["chiir", "short"],         paper_type="Short Paper"),
    CategoryRule(field="track name",  contains=["tutorial"],                   paper_type="Tutorial"),
    CategoryRule(field="track name",  contains=["workshop"],                   paper_type="Workshop"),
    CategoryRule(field="track name",  contains_all=["chiir", "pc"],            paper_type="Full Paper"),
]

config = ConferenceConfig(name=NAME, year=YEAR, accepted_decisions=ACCEPTED_DECISIONS,
                          paper_types=PAPER_TYPES, category_rules=CATEGORY_RULES)


def load(affiliation_mapping=None, **options):
    return load_easychair_data(EASYCHAIR_EXPORT, config, affiliation_mapping=affiliation_mapping, **options)


export = load()

display(pd.Series(export.decision_counts, name="submissions").to_frame()
        .assign(included=lambda df: [config.is_accepted(d) for d in df.index]))
display(pd.DataFrame([(t.section_name, t.original_name, t.paper_count) for t in export.tracks],
                     columns=["paper type", "EasyChair track(s)", "papers"]))

## 3. Fix the data before ACM sees it

Advice from previous CHIIR proceedings chairs:

- **TAPS prints the eRights data in the paper heading, whatever the camera-ready source says.** That covers affiliations, emails and addresses. If an author put a department, street address or two institutions into one field, that is what gets printed. Correcting it later means **removing the paper from TAPS and its eRights form**, after which the authors fill the form in again. Fixing it now is much cheaper.
- **Only one contact author gets ACM's emails.** Authors often tick several people as corresponding in EasyChair. Previously the last one ticked got the emails, and that was usually a busy senior author. This toolkit uses the **first** ticked author who has an email, and lists every paper where more than one was ticked so you can check who that is.

Affiliation canonicalisation (`affiliations_mapping.json`, e.g. "Adobe Inc." → "Adobe") is off by default, because it changes what authors typed. Turn it on only if you want consistent institution names, and check the `affiliation_canonicalised` notices it produces.

In [ ]:
AFFILIATION_MAPPING = None
# AFFILIATION_MAPPING = load_affiliation_mapping("affiliations_mapping.json")

export = load(AFFILIATION_MAPPING)

NEEDS_ACTION = ["uncategorised", "no_authors", "author_validation_error", "paper_validation_error",
                "affiliation_too_detailed", "multiple_corresponding", "invalid_email",
                "contact_author_priority_2", "contact_author_priority_3", "missing_email",
                "missing_affiliation", "author_order_mismatch", "name_consistency", "missing_approval_date"]

issues = pd.DataFrame([i.model_dump(exclude={"details"}) for i in export.validation_issues])
if issues.empty:
    print("No issues found")
else:
    issues["category"] = pd.Categorical(issues["category"], NEEDS_ACTION + sorted(set(issues["category"]) - set(NEEDS_ACTION)))
    with pd.option_context("display.max_rows", None, "display.max_colwidth", 120):
        display(issues[issues["category"].isin(NEEDS_ACTION)]
                .sort_values(["category", "paper_id"])[["category", "severity", "paper_id", "author_name", "message"]])
    print("Other notices (usually fine):", issues[~issues["category"].isin(NEEDS_ACTION)]["category"].value_counts().to_dict())

How to fix each kind of issue, in EasyChair. Then download the export again (step 1) and re-run steps 2–3.

| Category | What to do |
|---|---|
| `uncategorised` | Add a category rule in step 2, or fix the paper's form field or track |
| `affiliation_too_detailed` | Edit the author's affiliation to just the institution. Department, address and country don't belong there (country is its own field) |
| `multiple_corresponding` | Usually fine. The message names who will receive ACM's emails; untick the others if that person is wrong |
| `invalid_email`, `missing_email`, `contact_author_priority_*` | Get a working email for the contact author; without one, nobody will receive the eRights link |
| `missing_affiliation` | Ask the authors |
| `missing_approval_date` | Nothing yet; you set the dates in step 5 |
| `author_order_mismatch`, `name_consistency` | A name differs between the submission's author list and the author records (often a typo); make them match |

## 4. Ask authors to check their heading

Send each contact author the block for their paper (below), before you upload anything to eRights. Expect to send reminders several times; previous chairs still had 1–2 authors who didn't follow the instructions.

**Email template**

> **Subject:** CHIIR 2027 – please check your author details (paper #ID)
>
> Dear NAME,
>
> Congratulations again on your accepted paper. In the next few weeks ACM will email you, as the contact author, a link to the **eRights** copyright form, and then an invitation to upload your final version to **TAPS**.
>
> **Important:** TAPS builds the author block of the published paper from the eRights data, **not** from your LaTeX or Word source. Below is exactly what we will send to ACM. Please check the names, author order, affiliations and emails, and reply by DATE with any corrections.
>
> - The affiliation should be the institution only (e.g. "University of Glasgow"). Leave out the department, street address and postcode. Country is a separate field.
> - Use the same format if you edit the eRights form yourself.
> - If the heading is wrong after the form is submitted, we have to withdraw your paper from TAPS and eRights, and you will need to complete the form again.
> - Only the contact author receives ACM's emails, so please keep your co-authors informed.
>
> PREVIEW

In [ ]:
for preview in heading_previews(export):
    print(f"To: {preview['contact_name']} <{preview['contact_email']}>")
    print(preview["text"])
    print("-" * 80)

## 5. Create the eRights upload

- **ACM event ID** is in the eRights confirmation email from ACM, e.g. `3726522`.
- **Proceeding ID** is used in the XML header; use whatever the ACM coordinator gives you.
- **Dates** use the ACM format `DD-MON-YYYY`. Set both to the real submission deadline and notification date. Otherwise the dates are read from EasyChair, where the "last updated" time is *not* the approval date.

This writes one CSV per paper type plus `CHIIR-2027-All-Papers.csv` (the 36-column ACM Enhanced CSV, as used for CHIIR 2026) and the same data as ACM XML. Upload whichever format ACM asks for. Both are checked against the raw EasyChair export below.

In [ ]:
EVENT_ID = "PLACEHOLDER"           # from the eRights confirmation email
PROCEEDING_ID = f"{YEAR}-{NAME}"
SUBMISSION_DATE = "15-OCT-2026"    # DD-MON-YYYY
APPROVAL_DATE = "01-DEC-2026"
OUT_DIR = DATA_DIR / "output"

export = load(AFFILIATION_MAPPING, proceeding_id=PROCEEDING_ID,
              submission_date_override=SUBMISSION_DATE, approval_date_override=APPROVAL_DATE)

OUT_DIR.mkdir(parents=True, exist_ok=True)
xml_file = OUT_DIR / f"{NAME.lower()}{YEAR}.xml"
export_to_xml(export, str(xml_file))
problems = check_xml_against_source(EASYCHAIR_EXPORT, config, xml_file, affiliation_mapping=AFFILIATION_MAPPING)

if EVENT_ID.startswith("PLACEHOLDER"):
    print("Set EVENT_ID to write the eRights CSV files")
else:
    csv_files = export_to_erights_csv(export, OUT_DIR, EVENT_ID, config)
    problems += [p for path in csv_files.values() for p in check_erights_csv(path)]
    display(pd.DataFrame([(k, v.name) for k, v in csv_files.items()], columns=["paper type", "file"]))

print(f"XML: {xml_file}")
print("Cross-check:", "OK" if not problems else "")
for problem in problems:
    print("  ✗", problem)

Optional deeper analysis of the XML: one contact author per paper, similar affiliation spellings, authors who appear under two names, and the most common countries and institutions.

In [ ]:
from validate_acm_xml import validate_xml_file

validate_xml_file(str(xml_file), top_k=5);

Upload the CSV files (or the XML) at <https://erightsreview.acm.org> for your event. ACM then emails each contact author to complete their eRights form. Authors **must** finish eRights before TAPS will process their paper.

After eRights, authors get TAPS invitations and upload their LaTeX or Word source. Meanwhile you will need to:
- chase authors who miss the TAPS deadline
- review and approve papers that TAPS quality checks flag

The accepted-papers list for the website can be made now:

In [ ]:
from papers_to_docx import create_word_doc, extract_papers_from_xml

create_word_doc(extract_papers_from_xml(str(xml_file)), str(OUT_DIR / "accepted-papers.docx"), config)

## 6. Re-issuing one paper (only if a heading went wrong)

If a published heading is wrong after the paper is in TAPS:
1. Ask ACM to **remove the paper from TAPS and withdraw its eRights form**. The author can't change it until then.
2. Fix the data in EasyChair, download the export again (step 1) and re-run steps 2–3.
3. Set `PAPER_ID` below. It writes CSV/XML for that paper only, ready to upload again.

In [ ]:
PAPER_ID = None   # EasyChair submission number, e.g. 42

if PAPER_ID is not None:
    reissue = load(AFFILIATION_MAPPING, paper_id_filter=PAPER_ID, proceeding_id=PROCEEDING_ID,
                   submission_date_override=SUBMISSION_DATE, approval_date_override=APPROVAL_DATE)
    reissue_dir = OUT_DIR / f"reissue-{PAPER_ID}"
    reissue_dir.mkdir(parents=True, exist_ok=True)
    export_to_xml(reissue, str(reissue_dir / f"{NAME.lower()}{YEAR}-{PAPER_ID}.xml"))
    if not EVENT_ID.startswith("PLACEHOLDER"):
        export_to_erights_csv(reissue, reissue_dir, EVENT_ID, config)
    print(next(p["text"] for p in heading_previews(reissue)))
    print(f"\nWritten to {reissue_dir}")

## 7. Table of contents (after all papers are approved in TAPS)

In the TAPS admin panel, choose **Export Metadata** and save `ExportMetadata.xlsx` in `DATA_DIR`. First check that:
- `ORDER ID` gives the order you want in the proceedings (reorder in TAPS and export again if not)
- `PDFPAGES` is final; TAPS can show a provisional count before a paper is fully processed

Page numbers are cumulative, starting at 1.

In [ ]:
TAPS_EXPORT = DATA_DIR / "ExportMetadata.xlsx"

from generate_toc import generate_toc

if TAPS_EXPORT.exists():
    toc_text, toc = generate_toc(str(TAPS_EXPORT), str(OUT_DIR / f"{NAME.lower()}{YEAR}_toc"))
    display(toc)
else:
    print(f"Not yet: save the TAPS metadata export as {TAPS_EXPORT}")

Check the result: paper types in the expected order, plausible page counts (full papers are usually 10–12 pages, short papers 4), and no leftover HTML entities or garbled characters in author names.

## 8. Front matter

### Committee lists

Collect the PC/SPC spreadsheets from each track chair into `DATA_DIR / "referees"`. They can be EasyChair or OpenReview exports with columns such as first name, last name, email, affiliation and role. The scan reports duplicates and inconsistencies *across* files (the same person with a different affiliation or email, typos in names) and merges everything into `referees.xlsx`, with one sheet per track and role.

Fill in `REFEREE_LAYOUT` for this year's files:
- `file_tracks` gives each file's track label, and its role if the whole file is one role (e.g. `"PC"`, `"SPC"`)
- `file_roles` gives a role for files that have no role column
- `track_order` orders the sheets

In [ ]:
REFEREE_DIR = DATA_DIR / "referees"
REFEREE_LAYOUT = {
    "file_tracks": {
        # "full_papers_pc.xlsx": ["Full Papers", "PC"],
        # "full_papers_spc.xlsx": ["Full Papers", "SPC"],
    },
    "file_roles": {},
    "track_order": ["Full Papers", "Short Papers", "Demonstrations", "Doctoral Consortium"],
}

import scan_referees

if REFEREE_DIR.is_dir():
    scan_referees.configure(**REFEREE_LAYOUT)
    scan_referees.main(["--input", str(REFEREE_DIR), "--output", str(OUT_DIR / "referees.xlsx"),
                        "--report", str(OUT_DIR / "referees-report.html")])
else:
    print(f"Not yet: put the committee spreadsheets in {REFEREE_DIR}")

Fix any inconsistencies in the source spreadsheets and re-run the scan. Then choose which sheets of `referees.xlsx` go into the front matter, and their headings, in order. The output uses the ACM fonts (Linux Biolinum headings, Linux Libertine names) and is checked against the spreadsheet after writing.

In [ ]:
COMMITTEE_SECTIONS = [
    # "<sheet in referees.xlsx>=<heading>"
    # "Full Papers - SPC=Senior Program Committee:",
    # "Full Papers - PC=Program Committee:",
]

from referees_to_docx import load_sections, write_committee_docx

if COMMITTEE_SECTIONS and (OUT_DIR / "referees.xlsx").exists():
    write_committee_docx(load_sections(str(OUT_DIR / "referees.xlsx"), COMMITTEE_SECTIONS),
                         str(OUT_DIR / "committee.docx"))
else:
    print("Not yet: list COMMITTEE_SECTIONS once referees.xlsx is clean")

### Assemble and submit

Start from `templates/CHIIR26FrontMatter_Example.docx` and update it for this year:

| Section | What to fill in |
|---|---|
| Cover page | Conference name, year, dates, location, ISBN |
| Welcome messages | General chairs' and PC chairs' prefaces |
| Organising committee | Names and affiliations for each role |
| Programme committee | Paste from `committee.docx` |
| Sponsors | Logos and names |
| Table of contents | Paste from the `_toc.txt` file in step 7 and apply the `TOC Entry` paragraph style |

Tips:
- TOC page numbers refer to the proceedings PDF, not the Word document, so turn off Word's own page numbering for that section.
- Check the required format (PDF or `.docx`) and the deadline with the ACM proceedings coordinator.

## Optional: save these settings for the command line

This writes this year's conference settings to JSON, so the export can be repeated without the notebook (`easychair_export.py --config`).

In [ ]:
config_file = OUT_DIR / f"{NAME.lower()}{YEAR}.json"
config.to_json(config_file)
inputs = " ".join(map(str, EASYCHAIR_EXPORT)) if isinstance(EASYCHAIR_EXPORT, tuple) else str(EASYCHAIR_EXPORT)
print(f"Saved {config_file}\n")
print(f"python easychair_export.py --config {config_file} --input {inputs} --format csv "
      f"--event_id {EVENT_ID} --submission_date {SUBMISSION_DATE} --approval_date {APPROVAL_DATE} --output {OUT_DIR}")